# 02. Feature Engineering

KSPHM-KIMM 2025 Bearing RUL Prediction - Feature Engineering Notebook

This notebook covers:
1. Wavelet feature extraction (D4/D5 RMS, Entropy)
2. Envelope feature extraction (BPF RMS, Envelope RMS)
3. Feature visualization and analysis
4. Batch preprocessing pipeline

In [ ]:
import sys
sys.path.append('..')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from tqdm import tqdm

from src.preprocessing import (
    TDMSDataLoader,
    load_tdms_segments,
    WaveletFeatureExtractor,
    EnvelopeFeatureExtractor,
    extract_all_features
)
from src.utils import plot_feature_trends

## 1. Configuration

In [ ]:
# Configuration
DATA_DIR = "../data"
OUTPUT_DIR = "../data/preprocessed"

# Signal parameters
FS = 25600  # Sampling frequency (Hz)
WINDOW_SEC = 0.5
WINDOW_SIZE = int(FS * WINDOW_SEC)
CHANNELS = ["CH1", "CH2", "CH3", "CH4"]
ENV_TARGET_CHANNEL = "CH2"
BAND_RANGE = (1000, 5000)

print(f"Window size: {WINDOW_SIZE} samples ({WINDOW_SEC} seconds)")

## 2. Load Sample Data

In [ ]:
# Load single training set for demonstration
TRAIN_FOLDER = f"{DATA_DIR}/Train Set/Train1"

segments, timestamps = load_tdms_segments(TRAIN_FOLDER)
vib_data = {ch: np.concatenate([seg[ch].values for seg in segments]) for ch in CHANNELS}

print(f"Loaded {len(segments)} segments")
print(f"Total samples per channel: {len(vib_data['CH1']):,}")

## 3. Wavelet Feature Extraction

In [ ]:
from src.preprocessing.feature_extraction import extract_wavelet_params

# Extract wavelet features for a single window
sample_signal = vib_data["CH2"][:WINDOW_SIZE]
wavelet_params = extract_wavelet_params(sample_signal, wavelet="db4", level=5)

print("Wavelet Parameters (single window):")
for key, value in wavelet_params.items():
    print(f"  {key}: {value:.6f}")

In [ ]:
# Extract wavelet features for entire signal using class
wavelet_extractor = WaveletFeatureExtractor()

# Process CH2
wavelet_features = wavelet_extractor.extract(vib_data["CH2"], window_size=WINDOW_SIZE)

print(f"\nWavelet features extracted:")
for key, values in wavelet_features.items():
    print(f"  {key}: {len(values)} windows, range [{values.min():.4f}, {values.max():.4f}]")

In [ ]:
# Visualize wavelet features over time
time_axis = np.arange(len(wavelet_features["D4_RMS"])) * WINDOW_SEC

fig = plot_feature_trends(
    time_axis,
    wavelet_features,
    title="CH2 Wavelet Features Over Time"
)
plt.show()

## 4. Envelope Feature Extraction

In [ ]:
from src.preprocessing.feature_extraction import extract_envelope_params, EnvelopeConfig

# Extract envelope features for CH2
bpf_rms, envelope_rms = extract_envelope_params(
    vib_data["CH2"],
    fs=FS,
    lowcut=BAND_RANGE[0],
    highcut=BAND_RANGE[1],
    window_size=WINDOW_SIZE
)

print(f"BPF RMS shape: {bpf_rms.shape}")
print(f"Envelope RMS shape: {envelope_rms.shape}")
print(f"\nBPF RMS range: [{bpf_rms.min():.6f}, {bpf_rms.max():.6f}]")
print(f"Envelope RMS range: [{envelope_rms.min():.6f}, {envelope_rms.max():.6f}]")

In [ ]:
# Visualize envelope features
# Downsample for visualization (take one value per window)
n_windows = len(bpf_rms) // WINDOW_SIZE
bpf_windows = bpf_rms[::WINDOW_SIZE][:n_windows]
env_windows = envelope_rms[::WINDOW_SIZE][:n_windows]
time_axis = np.arange(len(bpf_windows)) * WINDOW_SEC

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(time_axis, bpf_windows)
axes[0].set_ylabel("BPF RMS")
axes[0].set_title("CH2 1000-5000Hz Band-pass Filtered RMS")
axes[0].grid(True, alpha=0.3)

axes[1].plot(time_axis, env_windows)
axes[1].set_ylabel("Envelope RMS")
axes[1].set_title("CH2 1000-5000Hz Envelope RMS")
axes[1].set_xlabel("Time (s)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Combined Feature Extraction

In [ ]:
# Extract all features at once
all_features = extract_all_features(
    vib_data,
    channels=CHANNELS,
    fs=FS,
    window_size=WINDOW_SIZE,
    env_target_channel=ENV_TARGET_CHANNEL,
    band_range=BAND_RANGE
)

print("Extracted features:")
for key, values in all_features.items():
    print(f"  {key}: shape {values.shape}")

## 6. Create Full Dataset

In [ ]:
def process_single_file(tdms_path, save_dir, config):
    """Process a single TDMS file and save features."""
    from nptdms import TdmsFile
    
    # Load TDMS
    tdms = TdmsFile.read(tdms_path)
    vib_group = tdms.groups()[0].name
    op_group = tdms.groups()[1].name
    
    row_count = len(tdms[vib_group]["CH1"].data)
    timestamps = (np.arange(0, row_count) / config["fs"]).astype(np.float32)
    
    # Extract vibration data
    vib_data = {
        ch: tdms[vib_group][ch].data[:row_count].astype(np.float32)
        for ch in config["channels"]
    }
    
    # Extract operation parameters
    op_dict = {ch.name.strip(): ch.data[0] for ch in tdms[op_group].channels()}
    
    # Build base DataFrame
    df = pd.DataFrame({
        "Time (s)": timestamps,
        **{ch: vib_data[ch] for ch in config["channels"]},
    })
    
    # Add operation parameters
    for key, col_name in [("torque", "Torque[Nm]"), ("front", "TC SP Front[C]"), ("rear", "TC SP Rear[C]")]:
        value = next((op_dict[k] for k in op_dict if key.lower() in k.lower()), np.nan)
        df[col_name] = np.repeat(value, row_count).astype(np.float32)
    
    # Extract and add features
    all_features = extract_all_features(
        vib_data,
        channels=config["channels"],
        fs=config["fs"],
        window_size=config["window_size"],
        env_target_channel=config["env_channel"],
        band_range=config["band_range"]
    )
    
    for key, values in all_features.items():
        df[key] = values[:row_count]
    
    # Save
    os.makedirs(save_dir, exist_ok=True)
    file_name = os.path.splitext(os.path.basename(tdms_path))[0] + ".csv"
    save_path = os.path.join(save_dir, file_name)
    df.to_csv(save_path, index=False)
    
    return save_path

In [ ]:
# Configuration for batch processing
config = {
    "fs": FS,
    "channels": CHANNELS,
    "window_size": WINDOW_SIZE,
    "env_channel": ENV_TARGET_CHANNEL,
    "band_range": BAND_RANGE
}

print("Configuration:")
for key, value in config.items():
    print(f"  {key}: {value}")

In [ ]:
# Process single file as example
import glob

sample_files = glob.glob(f"{TRAIN_FOLDER}/*.tdms")[:1]
if sample_files:
    test_output = f"{OUTPUT_DIR}/test"
    result = process_single_file(sample_files[0], test_output, config)
    print(f"Processed: {result}")
    
    # Load and display
    df_test = pd.read_csv(result)
    print(f"\nOutput shape: {df_test.shape}")
    print(f"Columns: {df_test.columns.tolist()}")
    display(df_test.head())

## 7. Feature Correlation Analysis

In [ ]:
# Calculate correlation between features
if 'df_test' in dir():
    feature_cols = [c for c in df_test.columns if 'RMS' in c or 'Entropy' in c]
    corr_matrix = df_test[feature_cols].corr()
    
    plt.figure(figsize=(12, 10))
    plt.imshow(corr_matrix, cmap='coolwarm', aspect='auto', vmin=-1, vmax=1)
    plt.colorbar(label='Correlation')
    plt.xticks(range(len(feature_cols)), feature_cols, rotation=90)
    plt.yticks(range(len(feature_cols)), feature_cols)
    plt.title('Feature Correlation Matrix')
    plt.tight_layout()
    plt.show()

## 8. Summary

### Selected Features

**Wavelet Features (per channel):**
- D4_RMS: RMS of D4 detail coefficients
- D5_RMS: RMS of D5 detail coefficients  
- D5_Entropy: Shannon entropy of D5 coefficients

**Envelope Features (CH2 only):**
- CH2_BPF_RMS: RMS of 1000-5000Hz band-pass filtered signal
- CH2_Envelope_RMS: RMS of envelope signal

### Rationale
- D4/D5 scales capture fault frequencies and harmonics (140Hz, 280Hz, etc.)
- 1000-5000Hz band isolates fault-related frequency content
- CH2 shows best correlation with degradation across training sets